# Class Imbalance and F1 Optimization Practice

**Topic:** Model Evaluation &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `X_train` `(6000, 12)`, `y_train`: fraud labels with ~17% positives.
- `X_val` `(2000, 12)`, `y_val`: a labeled validation set for threshold tuning.
- `X_test` `(2000, 12)` (predict these). `y_test` is used by the evaluation cell.

## Your task

1. Train a probabilistic classifier (logistic regression, boosting…), and consider `class_weight="balanced"`.
2. Use `X_val`/`y_val` to choose the probability threshold that maximizes F1.
3. Apply the tuned threshold to the test probabilities and set `predictions` to the resulting 0/1 labels.

## Evaluation

The evaluation cell reports precision, recall, and F1 on the test set, and F1 must reach the target. Accuracy is shown only to make a point.


In [ ]:
import numpy as np

rng = np.random.default_rng(131)
d = 12
w = rng.normal(0, 1.0, d)

def make(n, seed):
    r = np.random.default_rng(seed)
    X = r.normal(size=(n, d))
    logit = X @ w * 1.1 - 3.6          # rare positives
    y = (1 / (1 + np.exp(-logit)) > r.uniform(0, 1, n)).astype(int)
    return X, y

X_train, y_train = make(6000, 141)
X_val, y_val = make(2000, 142)
X_test, y_test = make(2000, 143)
print(f"positive rates: train {y_train.mean():.3f}, val {y_val.mean():.3f}")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: F1 on the test set (accuracy shown for contrast)
# =====================================================================
import numpy as np

TARGET_F1 = 0.55

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel().astype(int)
_true = np.asarray(y_test).ravel().astype(int)
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

_tp = int(((_pred == 1) & (_true == 1)).sum())
_fp = int(((_pred == 1) & (_true == 0)).sum())
_fn = int(((_pred == 0) & (_true == 1)).sum())
_prec = _tp / max(1, _tp + _fp)
_rec = _tp / max(1, _tp + _fn)
_f1 = 0.0 if _prec + _rec == 0 else 2 * _prec * _rec / (_prec + _rec)

print("=" * 52)
print(f"Accuracy  : {float((_pred == _true).mean()):.4f}   (all-negative scores {float((_true == 0).mean()):.4f}, so ignore accuracy!)")
print(f"Precision : {_prec:.4f}")
print(f"Recall    : {_rec:.4f}")
print(f"F1 score  : {_f1:.4f}   (target: >= {TARGET_F1})")
print("=" * 52)
print("PASSED: nice work!" if _f1 >= TARGET_F1 else "NOT PASSED yet: tune your threshold on the validation set.")


## Solution

Train a balanced logistic regression, then sweep the probability threshold on the validation set and keep the one with the best F1. Never tune the threshold on test. Apply it to the test probabilities.

```python
import numpy as np
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(class_weight="balanced", max_iter=2000)
clf.fit(X_train, y_train)

def f1_at(pred, true):
    tp = int(((pred == 1) & (true == 1)).sum())
    fp = int(((pred == 1) & (true == 0)).sum())
    fn = int(((pred == 0) & (true == 1)).sum())
    prec = tp / max(1, tp + fp)
    rec = tp / max(1, tp + fn)
    return 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)

p_val = clf.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0.05, 0.95, 181)
f1s = [f1_at((p_val >= t).astype(int), y_val) for t in thresholds]
best_t = float(thresholds[int(np.argmax(f1s))])
print(f"chosen threshold {best_t:.2f} (val F1 {max(f1s):.3f})")

predictions = (clf.predict_proba(X_test)[:, 1] >= best_t).astype(int)
```
